In [7]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import torch

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Load the dataset
df = pd.read_csv('Smishing_dataset1.csv')

# Clean the text
def clean_text(text):
    text = re.sub(r'[<][#][>]|<DECIMAL>', '', text)
    return text.strip()

df['Message'] = df['Message'].apply(clean_text)

# Convert labels to binary
df['Labels'] = df['Labels'].map({'ham': 0, 'smish': 1})

# Extract features using TF-IDF
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X = vectorizer.fit_transform(df['Message']).toarray()
y = df['Labels'].values

print("Feature matrix shape:", X.shape)
print("Label array shape:", y.shape)

Feature matrix shape: (11743, 1000)
Label array shape: (11743,)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity matrix
similarity_matrix = cosine_similarity(X)

# Construct incidence matrix H using k-NN (k=5)
k = 5
num_nodes = X.shape[0]
H = np.zeros((num_nodes, num_nodes))

for i in range(num_nodes):
    similar_indices = np.argsort(similarity_matrix[i])[-(k + 1):-1]
    H[similar_indices, i] = 1.0

# Compute degrees as vectors
Dv_vec = np.sum(H, axis=1)
De_vec = np.sum(H, axis=0)

# Convert to diagonal matrices
Dv = np.diag(Dv_vec ** -0.5)
De = np.diag(De_vec ** -1)
Dv[np.isinf(Dv)] = 0
De[np.isinf(De)] = 0

print("Incidence matrix shape:", H.shape)
print("Vertex degree matrix shape:", Dv.shape)
print("Edge degree matrix shape:", De.shape)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8924\3337660631.py:20: RuntimeWarning: divide by zero encountered in power
  Dv = np.diag(Dv_vec ** -0.5)


Incidence matrix shape: (11743, 11743)
Vertex degree matrix shape: (11743, 11743)
Edge degree matrix shape: (11743, 11743)


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Convert full matrices to tensors (assuming already defined from Step 2)
X_tensor = torch.FloatTensor(X)
H_tensor = torch.FloatTensor(H)  # Use dense H for now
y_tensor = torch.LongTensor(y)
Dv_tensor = torch.FloatTensor(Dv)
De_tensor = torch.FloatTensor(De)

class HGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=5, dropout=0.5):
        super(HGNN, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, hidden_dim))
        for _ in range(num_layers - 2):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
        self.layers.append(nn.Linear(hidden_dim, output_dim))
        self.dropout = dropout
        self.num_layers = num_layers

    def forward(self, X, H, Dv, De):
        batch_size = X.size(0)
        num_nodes = H.shape[0]

        # Slice for batch (use dense H)
        H_batch = H[:batch_size, :batch_size].to(X.device)
        Dv_batch = Dv[:batch_size, :batch_size].to(X.device)
        De_batch = De[:batch_size, :batch_size].to(X.device)

        # Validate shapes
        if H_batch.shape[0] != batch_size or H_batch.shape[1] != batch_size:
            raise ValueError(f"H_batch shape {H_batch.shape} does not match batch_size {batch_size}")
        if Dv_batch.shape != (batch_size, batch_size) or De_batch.shape != (batch_size, batch_size):
            raise ValueError(f"Dv_batch {Dv_batch.shape} or De_batch {De_batch.shape} mismatch with {batch_size}x{batch_size}")
        if X.shape[0] != batch_size:
            raise ValueError(f"X shape {X.shape} batch dimension {batch_size} mismatch")

        # Initial propagation
        X_out = torch.matmul(Dv_batch, X)
        X_out = torch.matmul(H_batch, torch.matmul(De_batch, torch.matmul(H_batch.T, X_out)))
        X_out = self.layers[0](X_out)
        X_out = F.relu(X_out)
        X_out = F.dropout(X_out, self.dropout, training=self.training)

        # Store the first layer output for residual
        prev_X = X_out

        # Layer-wise propagation with residuals for intermediate layers
        for i in range(1, len(self.layers) - 1):  # Start from second layer, exclude last
            X_in = prev_X
            X_out = self.layers[i](X_out)
            X_out = X_out + F.relu(X_in)  # Residual connection
            X_out = F.dropout(X_out, self.dropout, training=self.training)
            X_out = F.relu(X_out)
            prev_X = X_out.clone()  # Update prev_X for next iteration

        # Final layer (no residual)
        X_out = self.layers[-1](X_out)
        X_out = F.dropout(X_out, self.dropout, training=self.training)

        return F.log_softmax(X_out, dim=1)

# Initialize model with a subset for testing
model = HGNN(input_dim=X_tensor.shape[1], hidden_dim=64, output_dim=2, num_layers=5)
print(model)

# Test with a small batch to avoid crash
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
sample_size = 8  # Small batch size
sample_X = X_tensor[:sample_size].to(device)
sample_H = H_tensor[:sample_size, :sample_size].to(device)  # Use dense H
sample_Dv = Dv_tensor[:sample_size, :sample_size].to(device)
sample_De = De_tensor[:sample_size, :sample_size].to(device)
try:
    output = model(sample_X, sample_H, sample_Dv, sample_De)
    print("Model initialization and forward pass successful with sample batch!")
except Exception as e:
    print(f"Error during test forward pass: {e}")

HGNN(
  (layers): ModuleList(
    (0): Linear(in_features=1000, out_features=64, bias=True)
    (1-3): 3 x Linear(in_features=64, out_features=64, bias=True)
    (4): Linear(in_features=64, out_features=2, bias=True)
  )
)
Model initialization and forward pass successful with sample batch!


In [10]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.sparse

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42)

# Create data loaders with reduced batch size
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)  # Reduced to 16

# Compute class weights based on training data
class_counts = np.bincount(y_train.numpy())
num_samples = len(y_train)
base_weight = 1.0 / (class_counts / num_samples)
class_weights = torch.tensor([base_weight[0], max(base_weight[1], 3.0)], dtype=torch.float).to(device)
print(f"Class weights: {class_weights}")

# Convert H to sparse matrix to save memory
H_sparse = torch.sparse_coo_tensor(
    torch.where(H_tensor != 0),  # Non-zero indices
    H_tensor[torch.where(H_tensor != 0)],  # Non-zero values
    size=H_tensor.shape
)

# Helper functions
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        batch_size = X_batch.size(0)
        H_batch = H_sparse[:batch_size, :batch_size].to_dense().to(device)  # Convert to dense for batch
        Dv_batch = Dv_tensor[:batch_size, :batch_size].to(device)
        De_batch = De_tensor[:batch_size, :batch_size].to(device)
        print(f"Shapes - X_batch: {X_batch.shape}, H_batch: {H_batch.shape}, Dv_batch: {Dv_batch.shape}, De_batch: {De_batch.shape}")  # Debug
        optimizer.zero_grad()
        output = model(X_batch, H_batch, Dv_batch, De_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def evaluate(model, X_test, y_test, H_tensor, Dv_tensor, De_tensor, criterion, device):
    model.eval()
    with torch.no_grad():
        X_test, y_test = X_test.to(device), y_test.to(device)
        batch_size = X_test.size(0)
        H_test = H_sparse[:batch_size, :batch_size].to_dense().to(device)
        Dv_test = Dv_tensor[:batch_size, :batch_size].to(device)
        De_test = De_tensor[:batch_size, :batch_size].to(device)
        print(f"Shapes - X_test: {X_test.shape}, H_test: {H_test.shape}, Dv_test: {Dv_test.shape}, De_test: {De_test.shape}")  # Debug
        output = model(X_test, H_test, Dv_test, De_test)
        loss = criterion(output, y_test)
        probs = torch.softmax(output, dim=1)[:, 1]
        predicted = (probs > 0.3).int()
        return loss.item(), predicted.cpu().numpy(), y_test.cpu().numpy()

# Training setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(weight=class_weights)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, verbose=True)
best_loss = float('inf')
patience = 10
trigger_times = 0

# Training loop with early stopping
num_epochs = 100
for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, predicted, true = evaluate(model, X_test, y_test, H_sparse, Dv_tensor, De_tensor, criterion, device)
    scheduler.step(test_loss)
    if test_loss < best_loss:
        best_loss = test_loss
        trigger_times = 0
    else:
        trigger_times += 1
    if trigger_times >= patience:
        print(f'Early stopping at epoch {epoch+1}')
        break
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')

model.eval()

Class weights: tensor([1.1775, 6.6342])


TypeError: only integer tensors of a single element can be converted to an index

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Evaluate on test set
model.eval()
with torch.no_grad():
    X_test_tensor, y_test_tensor = X_test.to(device), y_test.to(device)
    H_test_tensor = H_sparse[:X_test.size(0), :X_test.size(0)].to_dense().to(device)
    Dv_test_tensor = Dv_tensor[:X_test.size(0), :X_test.size(0)].to(device)
    De_test_tensor = De_tensor[:X_test.size(0), :X_test.size(0)].to(device)
    output = model(X_test_tensor, H_test_tensor, Dv_test_tensor, De_test_tensor)
    probs = torch.softmax(output, dim=1)[:, 1]
    predicted = (probs > 0.3).int().cpu()
    accuracy = accuracy_score(y_test.cpu(), predicted)
    precision = precision_score(y_test.cpu(), predicted)
    recall = recall_score(y_test.cpu(), predicted)
    f1 = f1_score(y_test.cpu(), predicted)
    print(f'Test Accuracy: {accuracy:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1-Score: {f1:.4f}')

Test Accuracy: 0.6624
Precision: 0.2741
Recall: 0.7959
F1-Score: 0.4078
